# Train the conditional Groove VAE

This notebook loads `groove_training_pairs_genre.h5`, trains the conditional
VAE, and writes the best checkpoint to `best_conditional_groove_vae.pt`.


In [ ]:
# Input and output paths
TRAINING_PAIRS_PATH = "groove_training_pairs_genre.h5"
MODEL_OUTPUT_PATH = "text2groove_model.pt"


In [ ]:
import h5py
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pretty_midi
import random

from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
# Unique styles in the dataset:
# afrobeat
# afrocuban
# afrocuban/bembe
# afrocuban/calypso
# afrocuban/rhumba
# blues/shuffle
# country
# dance/breakbeat
# dance/disco
# funk
# funk/fast
# funk/latin
# funk/purdieshuffle
# funk/rock
# gospel
# highlife
# hiphop
# jazz
# jazz/fast
# jazz/funk
# jazz/fusion
# jazz/klezmer
# jazz/linear
# jazz/march
# jazz/mediumfast
# jazz/swing
# latin
# latin/bomba
# latin/bounce
# latin/brazilian
# latin/brazilian-baiao
# latin/brazilian-bossa
# latin/brazilian-frevo
# latin/brazilian-ijexa
# latin/brazilian-maracatu
# latin/brazilian-samba
# latin/brazilian-sambareggae
# latin/brazilian-songo
# latin/chacarera
# latin/dominican-merengue
# latin/merengue
# latin/reggaeton
# latin/samba
# latin/venezuelan-joropo
# latin/venezuelan-sangueo
# middleeastern
# neworleans/chacha
# neworleans/funk
# neworleans/secondline
# neworleans/shuffle
# pop
# pop/soft
# punk
# reggae
# reggae/slow
# rock
# rock/folk
# rock/halftime
# rock/indie
# rock/prog
# rock/rockabilly
# rock/shuffle
# soul
# soul/motown

GENRES = [
    "funk",
    "rock",
    "jazz",
    "hiphop",
]

FEEL_PHRASES = {
    "swung": [
        "with a swung feel",
        "with swing",
        "with delayed offbeats",
    ],
    "tight": [
        "with tight, precise timing",
        "locked to the grid",
        "with a straight pocket",
    ],
    "loose": [
        "with a relaxed pocket",
        "with laid-back timing",
        "with loose timing variation",
    ],
}

DYNAMICS_PHRASES = {
    "soft": [
        "play it softly",
        "use gentle dynamics",
        "keep the intensity low",
    ],
    "medium": [
        "use moderate dynamics",
        "keep the intensity balanced",
        "play with medium strength",
    ],
    "loud": [
        "play it loud",
        "use strong dynamics",
        "keep the intensity high",
    ],
}

PROMPT_PATTERNS = [
    "Give this a {genre} groove {feel}, {dynamics}.",
    "Humanize this as {genre} {feel}, {dynamics}.",
    "Make this sound like {genre} {feel}, {dynamics}.",
]


def _decode_h5_strings(values):
    # Decode a hdf5 string dataset to a np string array
    decoded = []
    for value in np.asarray(values):
        if isinstance(value, (bytes, np.bytes_)):
            decoded.append(value.decode("utf-8"))
        else:
            decoded.append(str(value))
    return np.asarray(decoded, dtype=str)


def create_attribute_prompt(genre, feel, overall_dynamics, random_prompt=True):
    # Build a prompt from the metadata attributes for one groove.
    # Validation/test use the first variant of each combination so repeated runs are deterministic.
    
    genre = str(genre).strip().lower() or "drum"
    feel = str(feel).strip().lower()
    overall_dynamics = str(overall_dynamics).strip().lower()

    chooser = random.choice if random_prompt else lambda values: values[0]

    return chooser(PROMPT_PATTERNS).format(
        genre=genre,
        feel=chooser(FEEL_PHRASES[feel]),
        dynamics=chooser(DYNAMICS_PHRASES[overall_dynamics]),
    )


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEXT_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

text_encoder = SentenceTransformer(TEXT_ENCODER_NAME)
text_encoder.to(device)
text_encoder.eval()

# Freeze all text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False

TEXT_EMB_DIM = text_encoder.get_embedding_dimension()

print("Device:", device)
print("Text embedding dim:", TEXT_EMB_DIM)

In [ ]:
# Encode the prompt strings produced by create_attribute_prompt.
# The cache avoids rerunning the frozen text encoder for wording variants
# that have already appeared in an earlier batch.
PROMPT_EMBEDDING_CACHE = {}


@torch.no_grad()
def encode_prompt_batch(prompts):
    if isinstance(prompts, str):
        prompts = [prompts]
    else:
        prompts = [str(prompt) for prompt in prompts]

    missing_prompts = list(dict.fromkeys(
        prompt
        for prompt in prompts
        if prompt not in PROMPT_EMBEDDING_CACHE
    ))

    if missing_prompts:
        missing_embeddings = text_encoder.encode(missing_prompts, batch_size=min(64, len(missing_prompts)),
            convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

        for prompt, embedding in zip(missing_prompts, missing_embeddings):
            PROMPT_EMBEDDING_CACHE[prompt] = torch.tensor(embedding, dtype=torch.float32)

    return torch.stack([PROMPT_EMBEDDING_CACHE[prompt] for prompt in prompts], dim=0).to(device)


In [ ]:
class GroovePromptDataset(Dataset):
    def __init__(self, h5_path, random_prompt=True):
        super().__init__()

        self.h5_path = h5_path
        self.random_prompt = random_prompt

        with h5py.File(h5_path, "r") as f:
            self.H = np.asarray(f["source"]["source_hits"], dtype=np.float32)
            self.O = np.asarray(f["target"]["target_offsets"], dtype=np.float32)
            self.V = np.asarray(f["target"]["target_velocities"], dtype=np.float32)

            metadata = f["metadata"]
            self.raw_styles = _decode_h5_strings(metadata["style"])
            self.feels = _decode_h5_strings(metadata["feel"])
            self.dynamics = _decode_h5_strings(metadata["overall_dynamic"])
            self.tempos = np.asarray(metadata["tempo"], dtype=np.float32)

        print("Loaded examples:", len(self.H))
        print("H shape:", self.H.shape)
        print("O shape:", self.O.shape)
        print("V shape:", self.V.shape)

    def __len__(self):
        return len(self.H)

    def __getitem__(self, idx):
        hit = self.H[idx]
        offset = self.O[idx]
        velocity = self.V[idx]
        tempo = self.tempos[idx]

        raw_style = self.raw_styles[idx]
        feel = self.feels[idx]
        overall_dynamics = self.dynamics[idx]

        prompt = create_attribute_prompt(genre=raw_style, feel=feel, overall_dynamics=overall_dynamics,
            random_prompt=self.random_prompt)

        # Loss should only apply where there is a hit.
        mask = hit.copy()

        return {
            "hits": torch.tensor(hit, dtype=torch.float32),
            "offsets": torch.tensor(offset, dtype=torch.float32),
            "velocities": torch.tensor(velocity, dtype=torch.float32),
            "mask": torch.tensor(mask, dtype=torch.float32),
            "prompt": prompt,
            "tempo": torch.tensor(tempo, dtype=torch.float32),
            "raw_style": raw_style,
            "feel": feel,
            "overall_dynamics": overall_dynamics
        }


In [ ]:
from torch.utils.data import Subset

h5_path = TRAINING_PAIRS_PATH
train_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=True)
print(len(train_base_dataset))
val_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=False)
test_base_dataset = GroovePromptDataset(h5_path=h5_path, random_prompt=False)
num_examples = len(train_base_dataset)

indices = np.arange(num_examples)
np.random.seed(42)
np.random.shuffle(indices)

val_fraction = 0.1
num_val = int(num_examples * val_fraction)
num_test = num_val * 2

val_indices = indices[:num_val]
test_indices = indices[num_val:num_test]
train_indices = indices[num_test:]

train_dataset = Subset(train_base_dataset, train_indices)
val_dataset = Subset(val_base_dataset, val_indices)
test_dataset = Subset(test_base_dataset, test_indices)

print("Train examples:", len(train_dataset))
print("Val examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

In [ ]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

train_labels = [
    train_base_dataset.raw_styles[i]
    for i in train_indices
]

train_counts = Counter(train_labels)

print("Train style counts:")
for style, count in train_counts.most_common():
    print(f"{style:15s} {count}")

alpha = 0.5

sample_weights = [
    1.0 / (train_counts[label] ** alpha)
    for label in train_labels
]

sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, 
                          shuffle=False, num_workers=0, drop_last=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

batch = next(iter(train_loader))

print("hits:", batch["hits"].shape)
print("offsets:", batch["offsets"].shape)
print("velocities:", batch["velocities"].shape)
print("mask:", batch["mask"].shape)
print("prompts:", batch["prompt"][:4])
print("tempo:", batch["tempo"].shape)
print("styles:", batch["raw_style"][:8])

In [ ]:
# Text-conditioned conditional groove VAE

# transform the Sentence Transformer embedding into the size expected by the groove model
class TextConditionProjection(nn.Module):
    def __init__(self, text_emb_dim, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(text_emb_dim),
            nn.Linear(text_emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

    def forward(self, text_emb):
        return self.net(text_emb)

class ConditionalGrooveVAE(nn.Module):
    """
    Training posterior:
        q(z | hits, target offsets, target velocities, text, tempo)

    Decoder:
        p(offsets, velocities | hits, text, tempo, z)

    At inference, z is sampled from N(0, I), so one hit pattern and prompt
    can produce multiple valid humanizations.
    """

    def __init__(self, text_emb_dim, d_model=192, latent_dim=32, n_heads=6, n_layers=3,
        dropout=0.1, n_steps=32, n_voices=9):
        super().__init__()

        self.n_steps = n_steps
        self.n_voices = n_voices
        self.n_tokens = n_steps * n_voices
        self.d_model = d_model
        self.latent_dim = latent_dim

        # Posterior encoder sees hit, target offset, and target velocity.
        self.performance_proj = nn.Linear(3, d_model)

        # Decoder sees only the fixed hit pattern.
        self.hit_proj = nn.Linear(1, d_model)

        self.step_emb = nn.Embedding(n_steps, d_model)
        self.voice_emb = nn.Embedding(n_voices, d_model)
        self.metric_emb = nn.Embedding(16, d_model)

        self.text_proj = TextConditionProjection(text_emb_dim=text_emb_dim, d_model=d_model)

        self.tempo_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )

        # Add the condition to every encoder token instead of relying on one
        # condition token that the Transformer can easily ignore.
        self.encoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.posterior_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.posterior_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.mu_head = nn.Linear(d_model, latent_dim)
        self.logvar_head = nn.Linear(d_model, latent_dim)

        self.latent_proj = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Each decoder token receives hit content, text/tempo condition, and z.
        self.decoder_condition_fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        # Non-autoregressive Transformer decoder where all 288 output
        # locations are predicted together because the hit score is fixed.
        self.performance_decoder = nn.TransformerEncoder(decoder_layer, num_layers=n_layers)

        self.offset_head = nn.Linear(d_model, 1)
        self.velocity_head = nn.Linear(d_model, 1)

        step_ids = []
        voice_ids = []
        metric_ids = []

        for step in range(n_steps):
            for voice in range(n_voices):
                step_ids.append(step)
                voice_ids.append(voice)
                metric_ids.append(step % 16)

        self.register_buffer(
            "step_ids",
            torch.tensor(step_ids, dtype=torch.long),
        )
        self.register_buffer(
            "voice_ids",
            torch.tensor(voice_ids, dtype=torch.long),
        )
        self.register_buffer(
            "metric_ids",
            torch.tensor(metric_ids, dtype=torch.long),
        )

        nn.init.normal_(self.posterior_token, std=0.02)

    def positional_tokens(self):
        return (
            self.step_emb(self.step_ids)[None, :, :]
            + self.voice_emb(self.voice_ids)[None, :, :]
            + self.metric_emb(self.metric_ids)[None, :, :]
        )

    def get_condition(self, prompt_emb, tempo):
        batch_size = prompt_emb.shape[0]

        text_condition = self.text_proj(prompt_emb)

        tempo_norm = (
            (tempo.view(batch_size, 1) - 120.0) / 40.0
        ).clamp(-3.0, 3.0)

        tempo_condition = self.tempo_proj(tempo_norm)

        return text_condition + tempo_condition

    def encode(
        self,
        hits,
        target_offsets,
        target_velocities,
        prompt_emb,
        tempo,
    ):
        batch_size = hits.shape[0]

        # Empty positions carry no target performance information.
        masked_offsets = target_offsets * hits
        masked_velocities = target_velocities * hits

        performance_features = torch.stack([hits, masked_offsets, masked_velocities],dim=-1).reshape(batch_size, self.n_tokens, 3)

        tokens = self.performance_proj(performance_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.encoder_condition_fusion(torch.cat([tokens, condition_tokens], dim=-1))
        posterior_token = (self.posterior_token.expand(batch_size, -1, -1) + condition[:, None, :])
        encoded = self.posterior_encoder(torch.cat([posterior_token, tokens], dim=1))

        posterior_state = encoded[:, 0, :]

        mu = self.mu_head(posterior_state)
        logvar = self.logvar_head(posterior_state).clamp(-8.0, 8.0)

        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, hits, prompt_emb, tempo, z):
        batch_size = hits.shape[0]

        hit_features = hits.reshape(batch_size, self.n_tokens, 1)

        tokens = self.hit_proj(hit_features)
        tokens = tokens + self.positional_tokens()

        condition = self.get_condition(prompt_emb, tempo)
        latent_condition = self.latent_proj(z)

        condition_tokens = condition[:, None, :].expand(-1, self.n_tokens, -1)
        latent_tokens = latent_condition[:, None, :].expand(-1, self.n_tokens, -1)

        tokens = self.decoder_condition_fusion(
            torch.cat(
                [tokens, condition_tokens, latent_tokens],
                dim=-1,
            )
        )

        decoded = self.performance_decoder(tokens)

        offsets = 0.5 * torch.tanh(
            self.offset_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        velocities = torch.sigmoid(
            self.velocity_head(decoded)
        ).reshape(batch_size, self.n_steps, self.n_voices)

        offsets = offsets * hits
        velocities = velocities * hits

        return offsets, velocities

    def forward(self, hits, target_offsets, target_velocities, prompt_emb, tempo, sample=True):
        mu, logvar = self.encode( hits=hits, target_offsets=target_offsets, target_velocities=target_velocities,
            prompt_emb=prompt_emb, tempo=tempo)

        z = (
            self.reparameterize(mu, logvar)
            if sample
            else mu
        )

        pred_offsets, pred_velocities = self.decode(hits=hits, prompt_emb=prompt_emb, tempo=tempo, z=z)

        return pred_offsets, pred_velocities, mu, logvar

In [ ]:
# Masked conditional-VAE objective

def masked_huber_loss(pred, target, mask):
    element_loss = F.smooth_l1_loss(pred, target, reduction="none")
    element_loss = element_loss * mask
    return element_loss.sum() / mask.sum().clamp_min(1.0)


def kl_divergence(mu, logvar):
    # Mean over the batch and latent dimensions keeps the KL scale stable
    # when latent_dim changes.
    kl_per_dimension = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
    return kl_per_dimension.mean()


def vae_groove_loss(pred_offsets, pred_velocities, target_offsets, target_velocities, mask, mu,
    logvar, beta, velocity_weight=0.5):
    
    offset_loss = masked_huber_loss(pred_offsets, target_offsets, mask)
    velocity_loss = masked_huber_loss(pred_velocities, target_velocities, mask)
    reconstruction_loss = (offset_loss + velocity_weight * velocity_loss)
    kl_loss = kl_divergence(mu, logvar)
    total_loss = reconstruction_loss + beta * kl_loss

    return {
        "loss": total_loss,
        "reconstruction_loss": reconstruction_loss,
        "offset_loss": offset_loss,
        "velocity_loss": velocity_loss,
        "kl_loss": kl_loss,
    }

In [ ]:
# Initialize conditional VAE

LATENT_DIM = 16

model = ConditionalGrooveVAE(
    text_emb_dim=TEXT_EMB_DIM,
    d_model=192,
    latent_dim=LATENT_DIM,
    n_heads=6,
    n_layers=3,
    dropout=0.1,
    n_steps=32,
    n_voices=9,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)

num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {num_params:,}")
print(f"Latent dimensions: {LATENT_DIM}")

In [ ]:
# Train and evaluation functions

from collections import defaultdict

def move_batch_to_device(batch, device):
    hits = batch["hits"].to(device)
    offsets = batch["offsets"].to(device)
    velocities = batch["velocities"].to(device)
    mask = batch["mask"].to(device)
    tempo = batch["tempo"].to(device)

    # batch["prompt"] contains the strings generated in cell 2.
    prompt_emb = encode_prompt_batch(batch["prompt"])

    return hits, offsets, velocities, mask, prompt_emb, tempo


def train_one_epoch_vae(model, loader, optimizer, device, beta):
    model.train()

    metric_sums = defaultdict(float)

    for batch in loader:
        hits, offsets, velocities, mask, prompt_emb, tempo = move_batch_to_device(batch, device)
        pred_offsets, pred_velocities, mu, logvar = model(
            hits=hits,
            target_offsets=offsets,
            target_velocities=velocities,
            prompt_emb=prompt_emb,
            tempo=tempo,
            sample=True,
        )

        losses = vae_groove_loss(
            pred_offsets=pred_offsets,
            pred_velocities=pred_velocities,
            target_offsets=offsets,
            target_velocities=velocities,
            mask=mask,
            mu=mu,
            logvar=logvar,
            beta=beta,
            velocity_weight=1,
        )

        optimizer.zero_grad(set_to_none=True)
        losses["loss"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        for name, value in losses.items():
            metric_sums[name] += value.item()

    return {
        name: total / len(loader)
        for name, total in metric_sums.items()
    }


@torch.inference_mode()
def evaluate_vae(model, loader, device, beta):
    model.eval()

    metric_sums = defaultdict(float)
    style_reconstruction_sum = defaultdict(float)
    style_count = defaultdict(int)

    for batch in loader:
        hits, offsets, velocities, mask, prompt_emb, tempo = move_batch_to_device(batch, device)

        # Use posterior mean during validation to remove sampling noise.
        pred_offsets, pred_velocities, mu, logvar = model(
            hits=hits,
            target_offsets=offsets,
            target_velocities=velocities,
            prompt_emb=prompt_emb,
            tempo=tempo,
            sample=False,
        )

        losses = vae_groove_loss(
            pred_offsets=pred_offsets,
            pred_velocities=pred_velocities,
            target_offsets=offsets,
            target_velocities=velocities,
            mask=mask,
            mu=mu,
            logvar=logvar,
            beta=beta,
            velocity_weight=1,
        )

        for name, value in losses.items():
            metric_sums[name] += value.item()

        element_offset = F.smooth_l1_loss(
            pred_offsets,
            offsets,
            reduction="none",
        )
        element_velocity = F.smooth_l1_loss(
            pred_velocities,
            velocities,
            reduction="none",
        )

        element_reconstruction = (element_offset + 1 * element_velocity) * mask
        per_example_reconstruction = (element_reconstruction.sum(dim=(1, 2)) / mask.sum(dim=(1, 2)).clamp_min(1.0))

        for style, example_loss in zip(batch["raw_style"], per_example_reconstruction.cpu().tolist()):
            style_reconstruction_sum[style] += example_loss
            style_count[style] += 1

    metrics = {
        name: total / len(loader)
        for name, total in metric_sums.items()
    }

    metrics["style_losses"] = {
        style: style_reconstruction_sum[style] / style_count[style]
        for style in style_reconstruction_sum
    }

    return metrics

In [ ]:
# Train conditional VAE

num_epochs = 200

# Start with a weak KL term and increase it gradually. A large KL weight from
# epoch 1 often causes posterior collapse in conditional sequence VAEs.
beta_max = 0.01
kl_warmup_epochs = 15
validation_interval = 5

best_val_objective = float("inf")
best_model_path = MODEL_OUTPUT_PATH

history = []

for epoch in range(1, num_epochs + 1):
    beta = beta_max * min(1.0, (epoch - 1) / max(1, kl_warmup_epochs))

    train_metrics = train_one_epoch_vae(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        device=device,
        beta=beta,
    )

    print(f"\nEpoch {epoch:03d} | beta={beta:.5f}")
    print(
        f"train total={train_metrics['loss']:.5f} | "
        f"recon={train_metrics['reconstruction_loss']:.5f} | "
        f"offset={train_metrics['offset_loss']:.5f} | "
        f"velocity={train_metrics['velocity_loss']:.5f} | "
        f"KL={train_metrics['kl_loss']:.5f}"
    )

    if epoch % validation_interval != 0 and epoch != num_epochs:
        continue

    val_metrics = evaluate_vae(model=model, loader=val_loader, device=device, beta=beta)

    history.append(
        {
            "epoch": epoch,
            "beta": beta,
            "train": train_metrics,
            "val": val_metrics,
        }
    )
    
    print(
        f"val   total={val_metrics['loss']:.5f} | "
        f"recon={val_metrics['reconstruction_loss']:.5f} | "
        f"offset={val_metrics['offset_loss']:.5f} | "
        f"velocity={val_metrics['velocity_loss']:.5f} | "
        f"KL={val_metrics['kl_loss']:.5f}"
    )

    print("val reconstruction by style:")
    for style, loss_value in sorted(
        val_metrics["style_losses"].items()
    ):
        print(f"  {style:15s}: {loss_value:.5f}")

    # Only select checkpoints after KL warm-up. Prior samples are unreliable
    # before q(z|x,c) has been regularized toward N(0, I).
    selection_objective = (val_metrics["reconstruction_loss"] + beta_max * val_metrics["kl_loss"])

    if (epoch >= kl_warmup_epochs and selection_objective < best_val_objective):
        best_val_objective = selection_objective

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "best_val_objective": best_val_objective,
                "val_reconstruction": val_metrics["reconstruction_loss"],
                "val_kl": val_metrics["kl_loss"],
                "beta": beta,
                "latent_dim": LATENT_DIM,
                "prompt_config": {
                    "patterns": PROMPT_PATTERNS,
                    "feel_phrases": FEEL_PHRASES,
                    "dynamics_phrases": DYNAMICS_PHRASES,
                },
                "model_config": {
                    "d_model": 192,
                    "latent_dim": LATENT_DIM,
                    "n_heads": 6,
                    "n_layers": 3,
                    "dropout": 0.1,
                    "n_steps": 32,
                    "n_voices": 9,
                },
            },
            best_model_path,
        )

        print(f"Saved new best model to {best_model_path}")